## Installing Libraries

In [ ]:
!pip install -q langchain_community
!pip install -q faiss-cpu
!pip install -q langchain-huggingface
!pip install -q langchain_openai

## Importing Libraries

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import os
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent

## Load Text Document

In [ ]:
loader = TextLoader("clinical_note_example.txt")
document = loader.load()
print("Number of documents loaded: ", len(document))
print("Content: ", document[0].page_content)
print("Metadata: ", document[0].metadata)

Number of documents loaded:  1
Content:  Clinical Note

Patient Name: John Doe DOB: 01/15/1975 Age: 50 years Sex: Male Date of
Visit: 03/01/2026 Provider: Dr. Jane Smith

Chief Complaint: Chest pain and shortness of breath for 2 days.

History of Present Illness: The patient is a 50-year-old male who presents with a 2-day history of intermittent chest pain described as pressure-like and located in the central chest. The pain occasionally radiates to the left shoulder and is associated with mild shortness of breath and diaphoresis. Symptoms began while performing moderate physical activity and have occurred several times since onset. No syncope reported. The patient rates the pain as 6/10 in severity.

Past Medical History: Hypertension Hyperlipidemia

Medications: Lisinopril 10 mg daily Atorvastatin 20 mg nightly

Allergies: No known drug allergies

Family History: Father with coronary artery disease diagnosed at age 60. Mother with type 2 diabetes.

Social History: Former smoker (quit

## Split the Document into Chunks

In [ ]:
doc_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
split_texts = doc_splitter.split_documents(document)
print(len(split_texts))

27


In [ ]:
split_texts[0].page_content

'Clinical Note'

In [ ]:
split_texts[1].page_content

'Patient Name: John Doe DOB: 01/15/1975 Age: 50 years Sex: Male Date of'

## Initialize the Text Embedding Model

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Build a FAISS Vector Store

In [ ]:
vectorstore=FAISS.from_documents(split_texts, embeddings)

In [ ]:
vectorstore

## Create a Retriever

In [ ]:
retriever=vectorstore.as_retriever()

## Create a Tool for the Agent

In [ ]:
@tool
def clinical_note_search(query: str) -> str:
    """
    Search the clinical note for relevant patient information.
    Use this tool when answering questions about the patient's symptoms,
    medical history, medications, diagnosis, or treatment plan.
    """
    docs = retriever.invoke(query)
    context_parts = []
    for doc in docs:
        context_parts.append(doc.page_content)
    return "\n\n".join(context_parts)

## Initialize the Large Language Model

In [ ]:
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY_HERE"
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0)

## Create the Agent

In [ ]:
agent = create_agent(
    model=llm,
    tools=[clinical_note_search],
    system_prompt=(
        """You are an assistant for question-answering tasks.
           Use the clinical_note_search tool whenever the answer should come from a clinical note.
           If you don't know the answer, just say that you don't know.
           Use one sentence and keep the answer concise.
        """
    ),
)

## Ask a Question

In [ ]:
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is the chief complaint of the patient?"}
        ]
    }
)

In [ ]:
result

{'messages': [HumanMessage(content='What is the chief complaint of the patient?', additional_kwargs={}, response_metadata={}, id='d3b38022-eced-415e-9f12-43e4203c70c0'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 139, 'total_tokens': 157, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSe1asIlOL5R4nTUtUDaFqgbGxZok', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d711f-e8c6-7f40-a240-92db695663a9-0', tool_calls=[{'name': 'clinical_note_search', 'args': {'query': 'chief complaint'}, 'id': 'call_EyploOk1L8P0WsCskKfIwXzD', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input

In [ ]:
print(result["messages"][-1].content)

The chief complaint of the patient is chest pain and shortness of breath for 2 days.


In [ ]:
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is the past medical history of the patient?"}
        ]
    }
)

In [ ]:
result

{'messages': [HumanMessage(content='What is the past medical history of the patient?', additional_kwargs={}, response_metadata={}, id='2bb56711-77c5-4be5-af7b-7c70609f124c'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 140, 'total_tokens': 159, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSe1oPrhk8Uz15kkGVhcaqovWrw50', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d7120-216b-71e1-a7f8-9d84ce256268-0', tool_calls=[{'name': 'clinical_note_search', 'args': {'query': 'past medical history'}, 'id': 'call_zyiFW8xBxPAnKCidGSCEHhSf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada

In [ ]:
print(result["messages"][-1].content)

The patient's past medical history includes hypertension and hyperlipidemia.


In [ ]:
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is the race of the patient?"}
        ]
    }
)

In [ ]:
result

{'messages': [HumanMessage(content='What is the race of the patient?', additional_kwargs={}, response_metadata={}, id='00b269c8-ff97-4e71-9295-a57e7b2da4dd'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 138, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSe1w1Uo1BE0tNJ4TLHzACNjIpLpf', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d7120-4047-7ae0-888d-3f9d3600544a-0', tool_calls=[{'name': 'clinical_note_search', 'args': {'query': 'patient race'}, 'id': 'call_DPMLKJLBdY5EZBl6btkdfMGL', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 138,

In [ ]:
print(result["messages"][-1].content)

The race of the patient is not mentioned in the clinical note.
